In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [2]:
import torch
import evaluate
import numpy as np
from peft import UIOrthoLoRAConfig, UILinLoRAConfig, get_peft_model, TaskType
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from datasets import load_dataset

In [3]:
def load_and_prepare(tokenizer, max_length=128):
    """Load E2E dataset and prepare tokenised fields."""
    ds = load_dataset("tuetschek/e2e_nlg")

    def linearise(record):
        mr = record["meaning_representation"]  # e.g. "name[Bibimbap House], food[Indian]"
        ref = record["human_reference"] if "human_reference" in record else record["reference"]
        prompt = f"{mr} => "  # simple prompt pattern
        example = prompt + ref
        tokenised = tokenizer(
            example,
            truncation=True,
            max_length=max_length,
            padding="max_length",
        )
        labels = tokenised["input_ids"].copy()
        # Mask prompt tokens so they are ignored in the loss (label = -100)
        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        prompt_len = len(prompt_ids)
        labels[:prompt_len] = [-100] * prompt_len
        tokenised["labels"] = labels
        return tokenised

    ds = ds.map(linearise, remove_columns=ds["train"].column_names)
    return ds

In [4]:
from pycocoevalcap.cider.cider import Cider

class CiderMetric:
    """Wraps pycocoevalcap so it looks like an `evaluate` metric."""
    def __init__(self):
        self.scorer = Cider()

    def compute(self, *, predictions, references):
        # pycocoevalcap expects dicts: {idx: ["sentence"]}
        hyps = {i: [pred] for i, pred in enumerate(predictions)}
        refs = {i: [ref]  for i, ref  in enumerate(references)}
        score, _ = self.scorer.compute_score(refs, hyps)
        return {"cider": score}

cider_metric = CiderMetric()

In [5]:
bleu_metric = evaluate.load("sacrebleu")
meteor_metric = evaluate.load("meteor")
rouge_metric = evaluate.load("rouge")
nist_metric = evaluate.load("nist_mt")

[nltk_data] Downloading package wordnet to /home/guyb/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to /home/guyb/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/guyb/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [6]:
def postprocess_text(preds, labels):
    preds = [p.strip() for p in preds]
    labels = [l.strip() for l in labels]
    return preds, labels


def set_contiguous(model):
    for m in model.modules():
        if hasattr(m, "parametrizations") and "weight" in m.parametrizations:
            base = m.parametrizations.weight[0].base
            if not base.is_contiguous():
                base.data = base.data.contiguous()


import numpy as np

import numpy as np

from transformers.trainer_utils import EvalPrediction

# ------------------------------------------------------------------
# helper -----------------------------------------------------------
def _postprocess_strs(predictions, references):
    """Strip leading/trailing spaces & unify whitespace."""
    preds = [p.strip() for p in predictions]
    refs  = [r.strip() for r in references]
    return preds, refs
# ------------------------------------------------------------------

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    # ───── tensor → numpy ────────────────────────────────────────────────────
    if isinstance(preds, torch.Tensor):
        preds = preds.cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()

    # ───── logits → ids (if needed) ──────────────────────────────────────────
    if preds.ndim == 3:             # (batch, seq_len, vocab)
        preds = preds.argmax(-1)

    # ───── mask ignored tokens in labels ─────────────────────────────────────
    labels = labels.copy()
    labels[labels == -100] = tokenizer.pad_token_id

    # ───── decode ────────────────────────────────────────────────────────────
    pred_strs  = tokenizer.batch_decode(preds,   skip_special_tokens=True)
    label_strs = tokenizer.batch_decode(labels,  skip_special_tokens=True)

    pred_strs  = [t.strip() for t in pred_strs]
    label_strs = [t.strip() for t in label_strs]

    # ───── metrics ───────────────────────────────────────────────────────────
    bleu   = bleu_metric.compute(
                predictions=pred_strs,
                references=[[r] for r in label_strs]
             )["score"]

    meteor = meteor_metric.compute(
                predictions=pred_strs,
                references=label_strs
             )["meteor"]

    rougeL = rouge_metric.compute(
                predictions=pred_strs,
                references=label_strs,
                use_stemmer=True
             )["rougeL"]

    # -------- NIST (handle both key names) -----------------------------------
    nist_raw = nist_metric.compute(
                predictions=pred_strs,
                references=label_strs
              )
    nist = nist_raw.get("nist", nist_raw.get("score"))

    return {
        "bleu":   round(bleu,   4),
        "meteor": round(meteor, 4),
        "rougeL": round(rougeL, 4),
        "nist":   round(nist,   4),
    }




In [7]:
orthoLoRAConfig = UIOrthoLoRAConfig(
    target_modules=["attn.c_attn", "attn.c_proj"],
    fan_in_fan_out         = True,   # GPT-2 matrices are (out, in)
    initial_scaler         = 0.1,    # scale of the diagonal Σ at init
    initial_sigma          = 0.1,    # std-dev for the trainable Σ entries
    uiortholora_alpha      = 1,
    uiortholora_dropout    = 0,
    num_svalues_to_adapt   = 2,       # adapt the top-4 singular values
    num_svectors_to_adapt  = 2,       # adapt the corresponding vectors
    task_type              = TaskType.CAUSAL_LM
)

In [8]:
from peft import LoraConfig

lora_config = LoraConfig(
    target_modules=["attn.c_attn", "attn.c_proj"],
    r=2,                         # very low rank for easy debugging
    lora_alpha=1,               # no extra scaling
    lora_dropout=0.0,           # no dropout for deterministic behavior
    bias="none",                # keep bias untouched
    fan_in_fan_out=False,       # match GPT-2 shape: (out, in)
    task_type=TaskType.CAUSAL_LM
)


In [9]:
model_path = "gpt2-medium"
seed=42

torch.manual_seed(seed)
np.random.seed(seed)


tokenizer = AutoTokenizer.from_pretrained(model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

ds = load_and_prepare(tokenizer)

base_model = AutoModelForCausalLM.from_pretrained(model_path)
base_model.config.pad_token_id = tokenizer.pad_token_id

/opt/anaconda3/envs/guyb_env2/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [10]:
model = get_peft_model(base_model, orthoLoRAConfig)

In [11]:
set_contiguous(model)
model.print_trainable_parameters()

trainable params: 147,936 || all params: 354,971,104 || trainable%: 0.0417


In [12]:
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="outputs/check",
    overwrite_output_dir=True,
    eval_strategy="no",
    save_strategy="no",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_accumulation_steps=2,
    learning_rate=1e-3,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=1,
    report_to="none",
)

In [13]:
trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=ds["train"].select(range(100)),
        eval_dataset=ds["validation"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

In [14]:
# trainer.train()
from accelerate import Accelerator
accelerator = Accelerator()
trainer = accelerator.prepare(trainer)
trainer.train()


/opt/anaconda3/envs/guyb_env2/lib/python3.11/site-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss


TrainOutput(global_step=35, training_loss=4.389678955078125, metrics={'train_runtime': 28.7277, 'train_samples_per_second': 17.405, 'train_steps_per_second': 1.218, 'total_flos': 116144394240000.0, 'train_loss': 4.389678955078125, 'epoch': 5.0})

In [15]:
trainer.save_model("outputs/models")

In [16]:
from pathlib import Path
import json

metrics = trainer.evaluate(ds["test"].select(range(100)))
Path(training_args.output_dir).mkdir(parents=True, exist_ok=True)
(Path(training_args.output_dir) / "test_metrics.json").write_text(json.dumps(metrics, indent=2))
print("Test metrics saved to", training_args.output_dir)


TypeError: type NoneType doesn't define __round__ method

In [ ]:
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")


In [15]:
# first_block = model.base_model.model.transformer.h[0]         # First transformer block
# attn = first_block.attn                                       # Attention module
# c_attn = attn.c_attn                                          # The fused QKV projection
# print(f"c_attn.weight shape: {c_attn.weight.shape}")          # Should be [3072, 1024]

c_attn.weight shape: torch.Size([1024, 3072])
